# Atlas companion: Dataset and Graph tutorials

This notebook teaches the same release-inspection task in two forms:

1. **Dataset tutorial** — discover the `data` API, filter the pinned catalog,
   resolve one recording, inspect its exact published files, and load its small
   retinotopy layer.
2. **Graph tutorial** — express those same operations as named functions with
   visible input/output ports, inspect the wiring, run the flow, and use an
   interactive recording selector.

Every example uses the published Zhong et al. release. It does not simulate a
figure or calculate a scientific result.

- [Article main text](https://www.nature.com/articles/s41586-025-09180-y#Sec1) · [Methods](https://www.nature.com/articles/s41586-025-09180-y#Sec9) ·
  [Data availability](https://www.nature.com/articles/s41586-025-09180-y#data-availability) ·
  [Code availability](https://www.nature.com/articles/s41586-025-09180-y#code-availability)
- [Published dataset, version 2](https://doi.org/10.25378/janelia.28811129.v2) ·
  [versioned Figshare API record](https://api.figshare.com/v2/articles/28811129/versions/2)


## Published-figure and method index

Use these links when moving from release mechanics to scientific analysis.

| Published item | Contents stated in the paper's figure caption | Analysis method |
|---|---|---|
| [Figure 1](https://www.nature.com/articles/s41586-025-09180-y#Fig1) | VR task and training timeline; licking; imaging; neural selectivity; cortical distributions and regional fractions | [Neural selectivity](https://www.nature.com/articles/s41586-025-09180-y#Sec20), [retinotopy](https://www.nature.com/articles/s41586-025-09180-y#Sec25) |
| [Figure 2](https://www.nature.com/articles/s41586-025-09180-y#Fig2) | Test-1 stimuli and licking; spatial sequences; coding-direction projections and similarity index | [Coding direction and similarity index](https://www.nature.com/articles/s41586-025-09180-y#Sec21) |
| [Figure 3](https://www.nature.com/articles/s41586-025-09180-y#Fig3) | Novel and adapted stimuli; selective-neuron distributions; coding-direction projections | [Neural selectivity](https://www.nature.com/articles/s41586-025-09180-y#Sec20), [coding direction and similarity index](https://www.nature.com/articles/s41586-025-09180-y#Sec21) |
| [Figure 4](https://www.nature.com/articles/s41586-025-09180-y#Fig4) | Rastermap view and the late-cue-versus-early-cue reward-prediction analysis | [Reward-prediction neurons](https://www.nature.com/articles/s41586-025-09180-y#Sec22) |
| [Figure 5](https://www.nature.com/articles/s41586-025-09180-y#Fig5) | Behavioural learning after natural-image, grating, or no pretraining | [Faster task learning after pretraining](https://www.nature.com/articles/s41586-025-09180-y#Sec7) |

Protocol details are in [imaging acquisition](https://www.nature.com/articles/s41586-025-09180-y#Sec13),
[visual stimuli](https://www.nature.com/articles/s41586-025-09180-y#Sec14), and
[behavioural training](https://www.nature.com/articles/s41586-025-09180-y#Sec17). Processing and inference details are in
[processing of calcium-imaging data](https://www.nature.com/articles/s41586-025-09180-y#Sec19) and
[statistics and reproducibility](https://www.nature.com/articles/s41586-025-09180-y#Sec24).


# Tutorial 1 — Dataset: find, inspect, and load published data

## 1. Connect to the fixed team workspace

The setup imports `drive.py` and `graph.py` from the exact shared-workspace
path below. It also checks the bundled, version-2 inventory before continuing.
The final line is intentionally just `data`: its notebook card lists the
variables available on the connected object and every public function you can
call.

Release provenance: [Figshare version 2](https://doi.org/10.25378/janelia.28811129.v2) and the paper's
[Data availability section](https://www.nature.com/articles/s41586-025-09180-y#data-availability).


In [ ]:
#@title Connect to the release { display-mode: "form" }
import importlib
import sys
import types
from pathlib import Path

from google.colab import drive as google_drive

google_drive.mount("/content/drive", force_remount=False)
WORKSPACE = Path(
    "/content/drive/MyDrive/Zhong et al. 2025 - Neuromatch Team Workspace"
)
required = (
    WORKSPACE / "drive.py",
    WORKSPACE / "graph.py",
    WORKSPACE / "zhong2025/atlas.py",
    WORKSPACE / "zhong2025/data.py",
    WORKSPACE / "zhong2025/assets/figshare-v2-inventory.json",
    WORKSPACE / "zhong2025/assets/imaging-experiment-index.json",
)
missing = [str(path) for path in required if not path.is_file()]
assert not missing, "Missing required workspace files:\n" + "\n".join(missing)

if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

# The shared Drive copy contains the data-access modules but not optional
# zhong2025 launcher/cache modules. Register the available package directory
# and expose only the helpers required by drive.py.
for module_name in list(sys.modules):
    if module_name in {"drive", "graph", "zhong2025"} or module_name.startswith("zhong2025."):
        sys.modules.pop(module_name, None)

zhong2025 = types.ModuleType("zhong2025")
zhong2025.__path__ = [str(WORKSPACE / "zhong2025")]
zhong2025.__package__ = "zhong2025"
sys.modules["zhong2025"] = zhong2025

atlas = importlib.import_module("zhong2025.atlas")
data_module = importlib.import_module("zhong2025.data")
for helper_name in (
    "experiment_rows",
    "format_bytes",
    "load_experiment_index",
    "load_file_inventory",
    "recording_bundle",
):
    setattr(zhong2025, helper_name, getattr(atlas, helper_name))
zhong2025.fetch_figshare_article = data_module.fetch_figshare_article

drive = importlib.import_module("drive")
graph = importlib.import_module("graph")
assert Path(drive.__file__).resolve() == (WORKSPACE / "drive.py").resolve()
assert Path(graph.__file__).resolve() == (WORKSPACE / "graph.py").resolve()

data = drive.setup(mount=False, report=False)
assert data.release["article_id"] == 28811129
assert data.release["version"] == 2
assert len(data.files) == 297
assert sum(item.size_bytes for item in data.files) == 452_233_500_962
data


## 2. Read the release-level variables

`data.connected`, `data.files`, `data.experiments`, and `data.folders` are
variables; `data.figshare(...)` is a function. Here `live=False` reads the
bundled snapshot of the [versioned Figshare API record](https://api.figshare.com/v2/articles/28811129/versions/2), so this
cell makes no network request and downloads no release file.


In [ ]:
release = data.figshare(live=False)
dataset_summary = {
    "connected": data.connected,
    "mounted_folders": data.folders,
    "published_files": len(data.files),
    "published_bytes": sum(item.size_bytes for item in data.files),
    "experiment_labels": len(data.experiments),
    "article_id": release["id"],
    "version": release["version"],
    "doi": release["doi"],
    "license": release["license"],
}
dataset_summary


In [ ]:
def inspect(instance):
  return { i: type(getattr(instance, i)) for i in dir(instance) if not i.startswith("_") }

In [ ]:
inspect(data)

In [ ]:
data.experiments

In [ ]:
{
    experiment: data.find(experiment = experiment)
    for experiment in data.experiments
}

In [ ]:
naive_test1 = data.load(
    data.find(experiment = 'naive_test1')[0]
)
naive_test1.keys()

In [ ]:
naive_test1['TX108_2023_01_05_2'].keys()

In [ ]:
naive_test1['TX108_2023_01_05_2']['LickTime']

In [ ]:
sup_train1_after_learning = data.load(
    data.find(experiment = 'sup_train1_after_learning')[0]
)
sup_train1_after_learning.keys()

In [ ]:
sup_train1_after_learning['VR2_2021_04_06_1'].keys()

In [ ]:
sup_train1_after_learning['VR2_2021_04_06_1']['LickTime']

In [ ]:
sup_train1_after_learning['VR2_2021_04_06_1']['ntrials']

In [ ]:
full_neural = data.find(category = 'full_neural')
len(full_neural), full_neural[0:5]

In [ ]:
reduced_neural = data.find(category = 'reduced_neural')
len(reduced_neural), reduced_neural[0:5]

In [ ]:
TX124_2023_12_24_1_neural_data = data.load(full_neural[0], max_gib=10)

In [ ]:
TX124_2023_12_24_1_neural_data

In [ ]:
len(TX124_2023_12_24_1_neural_data['spks'])

In [ ]:
[ i.shape for i in TX124_2023_12_24_1_neural_data['spks'] ]

In [ ]:
import numpy as np

if "TX124_2023_12_24_1_neural_data_spikes" not in globals():
    TX124_2023_12_24_1_neural_data_spikes = np.vstack(
        TX124_2023_12_24_1_neural_data["spks"]
    )
spikes = TX124_2023_12_24_1_neural_data_spikes


In [ ]:
TX124_2023_12_24_1_neural_data_spikes.shape

In [ ]:
import numpy as np

TX124_2023_12_24_1_SVD_dec = data.load(reduced_neural[0])
U = np.asarray(TX124_2023_12_24_1_SVD_dec["U"])
V = np.asarray(TX124_2023_12_24_1_SVD_dec["V"])
fs = 3.17


In [ ]:
TX124_2023_12_24_1_SVD_dec

In [ ]:
{ 'U': TX124_2023_12_24_1_SVD_dec['U'].shape, 'V': TX124_2023_12_24_1_SVD_dec['V'].shape }

In [ ]:
reconstructed_TX124_2023_12_24_1_neural_data_spikes = TX124_2023_12_24_1_SVD_dec['U'].T @ TX124_2023_12_24_1_SVD_dec['V']

In [ ]:
reconstructed_TX124_2023_12_24_1_neural_data_spikes.shape

In [ ]:
def plot_raw_neural_activity(
    spikes,
    start_frame=0,
    stop_frame=1000,
    first_neuron=0,
    number_of_neurons=100,
    fs=3.17,
):
    import numpy as np
    import matplotlib.pyplot as plt

    last_neuron = first_neuron + number_of_neurons

    activity = spikes[
        first_neuron:last_neuron,
        start_frame:stop_frame,
    ]

    time = np.arange(start_frame, stop_frame) / fs

    fig, ax = plt.subplots(figsize=(14, 6), constrained_layout=True)

    image = ax.imshow(
        activity,
        aspect="auto",
        origin="lower",
        cmap="gray_r",
        vmin=0,
        vmax=1,
        extent=[
            time[0],
            time[-1],
            first_neuron,
            last_neuron,
        ],
    )

    ax.set(
        title="Raw deconvolved neuronal activity",
        xlabel="Time (seconds)",
        ylabel="Neuron index",
    )

    fig.colorbar(image, ax=ax, label="Deconvolved activity")

    plt.show()
    return fig


raw_neural_figure = plot_raw_neural_activity(
    spikes,
    start_frame=0,
    stop_frame=200000,
    first_neuron=0,
    number_of_neurons=10,
)

In [ ]:
def plot_reconstructed_neural_activity(
    U,
    V,
    start_frame=0,
    stop_frame=1000,
    first_neuron=0,
    number_of_neurons=100,
    number_of_components=400,
    fs=3.17,
):
    import numpy as np
    import matplotlib.pyplot as plt

    last_neuron = first_neuron + number_of_neurons

    selected_U = U[
        :number_of_components,
        first_neuron:last_neuron,
    ]

    selected_V = V[
        :number_of_components,
        start_frame:stop_frame,
    ]

    reconstructed = selected_U.T @ selected_V
    time = np.arange(start_frame, stop_frame) / fs

    fig, ax = plt.subplots(figsize=(14, 6), constrained_layout=True)

    image = ax.imshow(
        reconstructed,
        aspect="auto",
        origin="lower",
        cmap="gray_r",
        vmin=0,
        vmax=1,
        extent=[
            time[0],
            time[-1],
            first_neuron,
            last_neuron,
        ],
    )

    ax.set(
        title=f"Neuronal activity reconstructed from "
              f"{number_of_components} SVD components",
        xlabel="Time (seconds)",
        ylabel="Neuron index",
    )

    fig.colorbar(image, ax=ax, label="Reconstructed activity")

    plt.show()
    return fig


reconstructed_neural_figure = plot_reconstructed_neural_activity(
    U,
    V,
    start_frame=0,
    stop_frame=1000,
    first_neuron=0,
    number_of_neurons=100,
    number_of_components=400,
    fs=fs,
)

In [ ]:
def plot_raw_vs_reconstructed_activity(
    spikes,
    U,
    V,
    start_frame=0,
    stop_frame=1000,
    first_neuron=0,
    number_of_neurons=100,
    number_of_components=400,
    fs=3.17,
):
    import numpy as np
    import matplotlib.pyplot as plt
    last_neuron = first_neuron + number_of_neurons
    raw = spikes[
        first_neuron:last_neuron,
        start_frame:stop_frame,
    ]
    reconstructed = (
        U[:number_of_components, first_neuron:last_neuron].T
        @ V[:number_of_components, start_frame:stop_frame]
    )
    time = np.arange(start_frame, stop_frame) / fs
    fig, axes = plt.subplots(
        2,
        1,
        figsize=(14, 10),
        sharex=True,
        constrained_layout=True,
    )
    image = axes[0].imshow(
        raw,
        aspect="auto",
        origin="lower",
        cmap="gray_r",
        vmin=0,
        vmax=1,
        extent=[
            time[0],
            time[-1],
            first_neuron,
            last_neuron,
        ],
    )
    axes[0].set(
        title="Raw neuronal activity",
        ylabel="Neuron index",
    )
    axes[1].imshow(
        reconstructed,
        aspect="auto",
        origin="lower",
        cmap="gray_r",
        vmin=0,
        vmax=1,
        extent=[
            time[0],
            time[-1],
            first_neuron,
            last_neuron,
        ],
    )
    axes[1].set(
        title=f"{number_of_components}-component SVD reconstruction",
        xlabel="Time (seconds)",
        ylabel="Neuron index",
    )
    fig.colorbar(
        image,
        ax=axes,
        label="Deconvolved activity",
        shrink=0.8,
    )
    plt.show()
    return fig
raw_vs_reconstructed_figure = plot_raw_vs_reconstructed_activity(
    spikes,
    U,
    V,
    start_frame=0,
    stop_frame=1000,
    first_neuron=0,
    number_of_neurons=100,
    number_of_components=400,
    fs=fs,
)

In [ ]:

tx124_sessions = data.recordings(mouse="TX124")
tx124_sessions

In [ ]:
tx124_sessions[0]

In [ ]:
tx124_sessions[0].files

In [ ]:
files = [
    (rec.recording_id, f.category, f.name, f.size_mib)
    for rec in tx124_sessions
    for f in rec.files
]
files

In [ ]:
labels = [f"{rid}\n{category}" for rid, category, _, _ in files]
sizes = [size for _, _, _, size in files]
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 6))
plt.barh(labels, sizes)
plt.xlabel("File size (MiB)")
plt.title("TX124 published data files")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
session = tx124_sessions[0]
experiment = "naive_test1"
retinotopy = session.load("retinotopy")
behavior = session.load("behavior", experiment=experiment)
def plot_retinotopy_map(retinotopy):
    import numpy as np
    import matplotlib.pyplot as plt

    xy = np.asarray(retinotopy["xy_t"], dtype=float)
    iarea = np.asarray(retinotopy["iarea"])

    area_groups = {
        "V1": (8,),
        "Medial": (0, 1, 2, 9),
        "Lateral": (5, 6),
        "Anterior": (3, 4),
    }

    colors = {
        "V1": "tab:blue",
        "Medial": "tab:green",
        "Lateral": "tab:orange",
        "Anterior": "tab:red",
    }

    fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
    assigned = np.zeros(len(iarea), dtype=bool)

    for name, codes in area_groups.items():
        selected = np.isin(iarea, codes)
        assigned |= selected

        ax.scatter(
            -xy[selected, 1],
            xy[selected, 0],
            s=3,
            alpha=0.4,
            color=colors[name],
            label=f"{name}: {selected.sum():,}",
            rasterized=True,
        )

    unassigned = ~assigned

    ax.scatter(
        -xy[unassigned, 1],
        xy[unassigned, 0],
        s=2,
        alpha=0.15,
        color="gray",
        label=f"Unassigned: {unassigned.sum():,}",
        rasterized=True,
    )

    ax.set(
        title=f"{session.recording_id}: retinotopy",
        xlabel="Cortical x = -xy_t[:, 1]",
        ylabel="Cortical y = xy_t[:, 0]",
    )
    ax.set_aspect("equal")
    ax.legend(markerscale=4, fontsize=8)

    plt.show()
    return fig

def plot_cortical_area_counts(retinotopy):
    import numpy as np
    import matplotlib.pyplot as plt

    iarea = np.asarray(retinotopy["iarea"])

    area_groups = {
        "V1": (8,),
        "Medial": (0, 1, 2, 9),
        "Lateral": (5, 6),
        "Anterior": (3, 4),
    }

    colors = {
        "V1": "tab:blue",
        "Medial": "tab:green",
        "Lateral": "tab:orange",
        "Anterior": "tab:red",
    }

    labels = list(area_groups)
    counts = [
        int(np.isin(iarea, area_groups[label]).sum())
        for label in labels
    ]

    fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)

    bars = ax.bar(
        labels,
        counts,
        color=[colors[label] for label in labels],
    )

    ax.bar_label(bars, labels=[f"{count:,}" for count in counts])

    ax.set(
        title=f"{session.recording_id}: neurons by cortical area",
        xlabel="Cortical-area group",
        ylabel="Number of neurons",
    )

    plt.show()
    return fig
def plot_lick_raster(behavior):
    import numpy as np
    import matplotlib.pyplot as plt

    lick_position = np.asarray(
        behavior.get("LickPos", ()),
        dtype=float,
    ) / 10.0

    lick_trial = np.asarray(
        behavior.get("LickTrind", ()),
        dtype=float,
    )

    sound_position = np.asarray(
        behavior.get("SoundPos", ()),
        dtype=float,
    ) / 10.0

    fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)

    number_of_licks = min(len(lick_position), len(lick_trial))

    if number_of_licks:
        ax.scatter(
            lick_position[:number_of_licks],
            lick_trial[:number_of_licks],
            s=6,
            alpha=0.5,
            label="Licks",
        )

    if len(sound_position):
        ax.scatter(
            sound_position,
            np.arange(len(sound_position)),
            marker="|",
            s=50,
            color="purple",
            label="Sound cue",
        )

    ax.axvline(
        4,
        color="gray",
        linestyle="--",
        linewidth=1,
        label="End of texture corridor",
    )

    ax.set(
        title=f"{session.recording_id}: lick raster — {experiment}",
        xlabel="Corridor position (m)",
        ylabel="Trial",
        xlim=(0, 6),
    )
    ax.legend(fontsize=8)

    plt.show()
    return fig
def plot_running_speed_by_position(behavior):
    import numpy as np
    import matplotlib.pyplot as plt

    position = np.asarray(
        behavior.get("ft_Pos", ()),
        dtype=float,
    ) / 10.0

    speed = np.asarray(
        behavior.get("ft_RunSpeed", ()),
        dtype=float,
    )

    number_of_frames = min(len(position), len(speed))

    position = position[:number_of_frames]
    speed = speed[:number_of_frames]

    valid = np.isfinite(position) & np.isfinite(speed)
    position = position[valid]
    speed = speed[valid]

    bins = np.linspace(0, 6, 31)
    centers = (bins[:-1] + bins[1:]) / 2
    bin_index = np.digitize(position, bins) - 1

    mean_speed = np.full(len(centers), np.nan)

    for index in range(len(centers)):
        selected = bin_index == index

        if np.any(selected):
            mean_speed[index] = np.mean(speed[selected])

    fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)

    ax.plot(
        centers,
        mean_speed,
        linewidth=2,
        color="tab:blue",
    )

    ax.axvline(
        4,
        color="gray",
        linestyle="--",
        linewidth=1,
    )

    ax.set(
        title=f"{session.recording_id}: running speed — {experiment}",
        xlabel="Corridor position (m)",
        ylabel="Mean running speed",
        xlim=(0, 6),
    )

    plt.show()
    return fig


running_speed_figure = plot_running_speed_by_position(behavior)

lick_figure = plot_lick_raster(behavior)

area_count_figure = plot_cortical_area_counts(retinotopy)

retinotopy_figure = plot_retinotopy_map(retinotopy)

In [ ]:
def plot_first_five_neuron_activities(
    spikes,
    start_frame=0,
    stop_frame=1000,
    fs=3.17,
):
    import numpy as np
    import matplotlib.pyplot as plt

    stop_frame = min(stop_frame, spikes.shape[1])
    neuron_activity = spikes[:5, start_frame:stop_frame]

    time = np.arange(start_frame, stop_frame) / fs

    fig, axes = plt.subplots(
        5,
        1,
        figsize=(14, 10),
        sharex=True,
        constrained_layout=True,
    )

    for neuron_index, ax in enumerate(axes):
        ax.plot(
            time,
            neuron_activity[neuron_index],
            color="black",
            linewidth=0.8,
        )

        ax.set_ylabel(f"Neuron {neuron_index}")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[-1].set_xlabel("Time (seconds)")
    axes[0].set_title("Activity of the first five neurons")

    plt.show()
    return fig


first_five_neurons_figure = plot_first_five_neuron_activities(
    spikes,
    start_frame=0,
    stop_frame=1000,
    fs=3.17,
)

In [ ]:
session = tx124_sessions[1]  # TX124_2023_12_24_1
session

In [ ]:
experiment = "naive_test3"
behavior_variant = "swap1"

behavior_file = session.file(
    "behavior",
    experiment=experiment,
)
behavior_bundle = data.load(behavior_file)

matching_keys = [
    key
    for key in behavior_bundle
    if str(key) == session.recording_id
    or str(key).startswith(session.recording_id + "_")
]
expected_key = f"{session.recording_id}_{behavior_variant}"
behavior_key = next(
    (key for key in matching_keys if str(key) == expected_key),
    None,
)
if behavior_key is None:
    raise KeyError(
        f"Behavior variant {expected_key!r} was not found; "
        f"available keys are {matching_keys}"
    )

behavior = behavior_bundle[behavior_key]

print("Loaded behavior key:", behavior_key)
print("Available variants:", matching_keys)
print("Behavior fields:", behavior.keys())


In [ ]:
experiment = "naive_test3"
behavior_variant = "swap1"

behavior_file = session.file(
    "behavior",
    experiment=experiment,
)
behavior_bundle = data.load(behavior_file)

matching_keys = [
    key
    for key in behavior_bundle
    if str(key) == session.recording_id
    or str(key).startswith(session.recording_id + "_")
]
expected_key = f"{session.recording_id}_{behavior_variant}"
behavior_key = next(
    (key for key in matching_keys if str(key) == expected_key),
    None,
)
if behavior_key is None:
    raise KeyError(
        f"Behavior variant {expected_key!r} was not found; "
        f"available keys are {matching_keys}"
    )

behavior = behavior_bundle[behavior_key]

print("Loaded behavior key:", behavior_key)
print("Available variants:", matching_keys)
print("Behavior fields:", behavior.keys())


In [ ]:
def plot_neuron_selectivity(
    spikes,
    behavior,
    retinotopy,
    threshold=0.3,
):
    import numpy as np
    import matplotlib.pyplot as plt

    spikes = np.asarray(spikes, dtype=float)

    # Align neural and behavioural frames.
    wall_id = np.asarray(behavior["ft_WallID"])
    moving = np.asarray(behavior["ft_move"]) > 0
    in_corridor = np.asarray(
        behavior["ft_CorrSpc"],
        dtype=bool,
    )

    number_of_frames = min(
        spikes.shape[1],
        len(wall_id),
        len(moving),
        len(in_corridor),
    )

    spikes = spikes[:, :number_of_frames]
    wall_id = wall_id[:number_of_frames]

    valid_frames = (
        moving[:number_of_frames]
        & in_corridor[:number_of_frames]
    )

    # Map the canonical stimulus roles to wall IDs.
    stimulus_id = np.asarray(behavior["stim_id"])
    unique_walls = np.asarray(behavior["UniqWalls"])

    leaf_walls = unique_walls[stimulus_id == 2]
    circle_walls = unique_walls[stimulus_id == 0]

    if len(leaf_walls) == 0:
        raise ValueError("No leaf1 stimulus was found")

    if len(circle_walls) == 0:
        raise ValueError("No circle1 stimulus was found")

    leaf_frames = (
        (wall_id == leaf_walls[0])
        & valid_frames
    )

    circle_frames = (
        (wall_id == circle_walls[0])
        & valid_frames
    )

    if leaf_frames.sum() < 2 or circle_frames.sum() < 2:
        raise ValueError(
            "Not enough valid leaf1 or circle1 frames"
        )

    # Responses for every neuron.
    leaf_activity = spikes[:, leaf_frames]
    circle_activity = spikes[:, circle_frames]

    mean_leaf = np.nanmean(leaf_activity, axis=1)
    mean_circle = np.nanmean(circle_activity, axis=1)

    std_leaf = np.nanstd(leaf_activity, axis=1)
    std_circle = np.nanstd(circle_activity, axis=1)

    denominator = std_leaf + std_circle

    # Paper-compatible signed d-prime.
    with np.errstate(divide="ignore", invalid="ignore"):
        dprime = (
            2.0
            * (mean_leaf - mean_circle)
            / denominator
        )

    finite = np.isfinite(dprime)

    leaf_selective = finite & (dprime >= threshold)
    circle_selective = finite & (dprime <= -threshold)
    nonselective = finite & ~(leaf_selective | circle_selective)

    # Retinotopy coordinates.
    xy = np.asarray(retinotopy["xy_t"], dtype=float)
    iarea = np.asarray(retinotopy["iarea"])

    number_of_neurons = min(
        len(dprime),
        len(xy),
        len(iarea),
    )

    dprime = dprime[:number_of_neurons]
    mean_leaf = mean_leaf[:number_of_neurons]
    mean_circle = mean_circle[:number_of_neurons]

    finite = finite[:number_of_neurons]
    leaf_selective = leaf_selective[:number_of_neurons]
    circle_selective = circle_selective[:number_of_neurons]
    nonselective = nonselective[:number_of_neurons]

    xy = xy[:number_of_neurons]
    iarea = iarea[:number_of_neurons]

    mapped = (iarea != -1) & (iarea != 7)

    leaf_selective &= mapped
    circle_selective &= mapped
    nonselective &= mapped
    finite &= mapped

    leaf_color = "tab:green"
    circle_color = "tab:purple"
    neutral_color = "lightgray"

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(14, 10),
        constrained_layout=True,
    )

    # 1. Selectivity distribution
    axes[0, 0].hist(
        dprime[finite],
        bins=60,
        color="gray",
        alpha=0.8,
    )

    axes[0, 0].axvline(
        threshold,
        color=leaf_color,
        linestyle="--",
        label=f"Leaf1: d′ ≥ {threshold}",
    )

    axes[0, 0].axvline(
        -threshold,
        color=circle_color,
        linestyle="--",
        label=f"Circle1: d′ ≤ −{threshold}",
    )

    axes[0, 0].set(
        title="Distribution of neuronal selectivity",
        xlabel="Signed d′: leaf1 − circle1",
        ylabel="Number of neurons",
    )
    axes[0, 0].legend()

    # 2. Mean responses
    axes[0, 1].scatter(
        mean_circle[nonselective],
        mean_leaf[nonselective],
        s=5,
        alpha=0.25,
        color=neutral_color,
        label="Non-selective",
    )

    axes[0, 1].scatter(
        mean_circle[leaf_selective],
        mean_leaf[leaf_selective],
        s=8,
        alpha=0.6,
        color=leaf_color,
        label="Leaf1-selective",
    )

    axes[0, 1].scatter(
        mean_circle[circle_selective],
        mean_leaf[circle_selective],
        s=8,
        alpha=0.6,
        color=circle_color,
        label="Circle1-selective",
    )

    axes[0, 1].set(
        title="Mean stimulus responses",
        xlabel="Mean circle1 activity",
        ylabel="Mean leaf1 activity",
    )
    axes[0, 1].legend(markerscale=2)

    # 3. Cortical selectivity map
    axes[1, 0].scatter(
        -xy[mapped, 1],
        xy[mapped, 0],
        s=2,
        alpha=0.15,
        color=neutral_color,
    )

    axes[1, 0].scatter(
        -xy[leaf_selective, 1],
        xy[leaf_selective, 0],
        s=8,
        alpha=0.65,
        color=leaf_color,
        label="Leaf1-selective",
    )

    axes[1, 0].scatter(
        -xy[circle_selective, 1],
        xy[circle_selective, 0],
        s=8,
        alpha=0.65,
        color=circle_color,
        label="Circle1-selective",
    )

    axes[1, 0].set(
        title="Cortical locations of selective neurons",
        xlabel="Cortical x = -xy_t[:, 1]",
        ylabel="Cortical y = xy_t[:, 0]",
    )
    axes[1, 0].set_aspect("equal")
    axes[1, 0].legend(markerscale=2)

    # 4. Selective neuron counts
    labels = [
        "Leaf1",
        "Circle1",
        "Non-selective",
    ]

    counts = [
        int(leaf_selective.sum()),
        int(circle_selective.sum()),
        int(nonselective.sum()),
    ]

    bars = axes[1, 1].bar(
        labels,
        counts,
        color=[
            leaf_color,
            circle_color,
            neutral_color,
        ],
    )

    axes[1, 1].bar_label(
        bars,
        labels=[f"{count:,}" for count in counts],
    )

    axes[1, 1].set(
        title=f"Selectivity classification, |d′| ≥ {threshold}",
        ylabel="Number of neurons",
    )

    fig.suptitle(
        f"{session.recording_id}: leaf1 versus circle1 selectivity\n"
        f"{leaf_frames.sum():,} leaf frames; "
        f"{circle_frames.sum():,} circle frames"
    )

    plt.show()

    return {
        "figure": fig,
        "dprime": dprime,
        "leaf_selective": leaf_selective,
        "circle_selective": circle_selective,
        "nonselective": nonselective,
    }


selectivity_result = plot_neuron_selectivity(
    spikes,
    behavior,
    retinotopy,
    threshold=0.3,
)

In [ ]:
tx108_sessions = data.recordings(mouse="TX108")
tx108_sessions

In [ ]:
[ [i.mouse, i.experiments] for i in tx108_sessions ]

In [ ]:
def plot_neuron_selectivity_before_after(
    data,
    mouse="TX108",
    before_experiment="sup_train1_before_learning",
    after_experiment="sup_train1_after_learning",
    threshold=0.3,
):
    import numpy as np
    import matplotlib.pyplot as plt
    from zhong2025.learning import svd_dprime

    area_codes = {
        "All": None,
        "V1": (8,),
        "Medial": (0, 1, 2, 9),
        "Lateral": (5, 6),
        "Anterior": (3, 4),
    }

    def find_recording(experiment):
        matches = [
            recording
            for recording in data.recordings(mouse=mouse)
            if experiment in recording.experiments
        ]

        if len(matches) != 1:
            raise ValueError(
                f"Expected one {mouse} recording for {experiment}; "
                f"found {[r.recording_id for r in matches]}"
            )

        return matches[0]

    def load_behavior(recording, experiment):
        behavior_file = recording.file(
            "behavior",
            experiment=experiment,
        )

        behavior_bundle = data.load(
            behavior_file,
            max_gib=1.0,
        )

        matching_keys = [
            key
            for key in behavior_bundle
            if str(key) == recording.recording_id
            or str(key).startswith(recording.recording_id + "_")
        ]

        if len(matching_keys) != 1:
            raise ValueError(
                f"Expected one behavior key for "
                f"{recording.recording_id}; found {matching_keys}"
            )

        return behavior_bundle[matching_keys[0]]

    def calculate_session_selectivity(recording, experiment):
        behavior = load_behavior(recording, experiment)
        svd = recording.load("reduced_neural", max_gib=1.0)
        retinotopy = recording.load("retinotopy", max_gib=1.0)

        U = np.asarray(svd["U"])
        V = np.asarray(svd["V"])

        wall_id = np.asarray(behavior["ft_WallID"])
        moving = np.asarray(behavior["ft_move"]) > 0
        in_corridor = np.asarray(
            behavior["ft_CorrSpc"],
            dtype=bool,
        )

        number_of_frames = min(
            V.shape[1],
            len(wall_id),
            len(moving),
            len(in_corridor),
        )

        V = V[:, :number_of_frames]
        wall_id = wall_id[:number_of_frames]

        valid_frames = (
            moving[:number_of_frames]
            & in_corridor[:number_of_frames]
        )

        stimulus_id = np.asarray(behavior["stim_id"])
        unique_walls = np.asarray(behavior["UniqWalls"])

        leaf_walls = unique_walls[stimulus_id == 2]
        circle_walls = unique_walls[stimulus_id == 0]

        if not len(leaf_walls) or not len(circle_walls):
            raise ValueError(
                f"{recording.recording_id} is missing "
                "leaf1 or circle1"
            )

        leaf_frames = (
            (wall_id == leaf_walls[0])
            & valid_frames
        )

        circle_frames = (
            (wall_id == circle_walls[0])
            & valid_frames
        )

        dprime = svd_dprime(
            U,
            V,
            leaf_frames,
            circle_frames,
        )

        iarea = np.asarray(retinotopy["iarea"])

        number_of_neurons = min(len(dprime), len(iarea))
        dprime = dprime[:number_of_neurons]
        iarea = iarea[:number_of_neurons]

        mapped = (iarea != -1) & (iarea != 7)
        finite = mapped & np.isfinite(dprime)
        selective = finite & (np.abs(dprime) >= threshold)

        fractions = {}

        for area_name, codes in area_codes.items():
            if codes is None:
                area_mask = finite
            else:
                area_mask = finite & np.isin(iarea, codes)

            fractions[area_name] = (
                np.mean(selective[area_mask])
                if np.any(area_mask)
                else np.nan
            )

        return {
            "recording_id": recording.recording_id,
            "dprime": dprime,
            "finite": finite,
            "selective": selective,
            "fractions": fractions,
            "leaf_frames": int(leaf_frames.sum()),
            "circle_frames": int(circle_frames.sum()),
        }

    before_recording = find_recording(before_experiment)
    after_recording = find_recording(after_experiment)

    before = calculate_session_selectivity(
        before_recording,
        before_experiment,
    )

    after = calculate_session_selectivity(
        after_recording,
        after_experiment,
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(14, 5),
        constrained_layout=True,
    )

    # Selectivity distributions
    combined = np.concatenate([
        before["dprime"][before["finite"]],
        after["dprime"][after["finite"]],
    ])

    limit = max(
        0.5,
        np.nanpercentile(np.abs(combined), 99),
    )

    bins = np.linspace(-limit, limit, 70)

    axes[0].hist(
        before["dprime"][before["finite"]],
        bins=bins,
        density=True,
        histtype="step",
        linewidth=2,
        label="Before training",
    )

    axes[0].hist(
        after["dprime"][after["finite"]],
        bins=bins,
        density=True,
        histtype="step",
        linewidth=2,
        label="After training",
    )

    axes[0].axvline(
        threshold,
        color="gray",
        linestyle="--",
    )
    axes[0].axvline(
        -threshold,
        color="gray",
        linestyle="--",
    )

    axes[0].set(
        title="Neuron-selectivity distributions",
        xlabel="Signed d′: leaf1 − circle1",
        ylabel="Density",
    )
    axes[0].legend()

    # Fraction selective by cortical area
    area_names = list(area_codes)
    x = np.arange(len(area_names))
    width = 0.38

    before_fraction = [
        before["fractions"][area]
        for area in area_names
    ]

    after_fraction = [
        after["fractions"][area]
        for area in area_names
    ]

    axes[1].bar(
        x - width / 2,
        before_fraction,
        width,
        label="Before training",
    )

    axes[1].bar(
        x + width / 2,
        after_fraction,
        width,
        label="After training",
    )

    axes[1].set(
        title=f"Fraction selective: |d′| ≥ {threshold}",
        xlabel="Cortical area",
        ylabel="Fraction of neurons",
        xticks=x,
        xticklabels=area_names,
    )
    axes[1].legend()

    fig.suptitle(
        f"{mouse}: neural selectivity before and after training\n"
        f"{before['recording_id']} → {after['recording_id']}"
    )

    plt.show()

    return {
        "figure": fig,
        "before": before,
        "after": after,
    }


selectivity_comparison = plot_neuron_selectivity_before_after(
    data,
    mouse="TX108",
    before_experiment="sup_train1_before_learning",
    after_experiment="sup_train1_after_learning",
    threshold=0.3,
)
plt.show()

In [ ]:
data

In [ ]:
mice = {
    "supervised": ['TX108', 'TX109', 'TX60', 'TX61', 'VR2'],
    "unsupervised": ['DR10', 'DR15', 'TX104', 'TX105', 'TX119', 'TX123', 'TX83', 'TX85', 'TX88'],
    "grating": ['LZ13', 'LZ16', 'TX139'],
    "naive": ['TX124', 'TX140'],
}
mice.items()

In [ ]:
mice_recordings = {
    category: {
      mouse: data.recordings(mouse = mouse)
    for mouse in elements
    }
  for (category, elements) in mice.items()
}
mice_recordings

In [ ]:
mice_recordings['supervised']

In [ ]:
mice_recordings['supervised']['TX60']

In [ ]:
import ipywidgets as widgets


EXPERIMENT_PAIRS = {
    "supervised": (
        "sup_train1_before_learning",
        "sup_train1_after_learning",
    ),
    "unsupervised": (
        "unsup_train1_before_learning",
        "unsup_train1_after_learning",
    ),
    "grating": (
        "train1_before_grating",
        "train1_after_grating",
    ),
}


@graph.node(outputs="selected_mice")
def select_three_mice(
    supervised_mouse="TX108",
    unsupervised_mouse="DR10",
    grating_mouse="LZ13",
):
    return {
        "supervised": supervised_mouse,
        "unsupervised": unsupervised_mouse,
        "grating": grating_mouse,
    }


@graph.node(
outputs=(
  "supervised_figure",
  "unsupervised_figure",
  "grating_figure",
), cache=False,
)
def plot_three_selected_mice(selected_mice, threshold=0.3):
    figures = []
    plt.close("all")
    for category in (
        "supervised",
        "unsupervised",
        "grating",
    ):
        before_experiment, after_experiment = (
            EXPERIMENT_PAIRS[category]
        )

        result = plot_neuron_selectivity_before_after(
            data,
            mouse=selected_mice[category],
            before_experiment=before_experiment,
            after_experiment=after_experiment,
            threshold=threshold,
        )

        figures.append(result["figure"])

    return tuple(figures)

In [ ]:
three_mouse_selectivity_graph = graph.Graph(
    "Compare one mouse from each cohort",
    select_three_mice,
    plot_three_selected_mice,
)

three_mouse_selectivity_panel = (
    three_mouse_selectivity_graph.widget(
        controls={
            "supervised_mouse": widgets.Dropdown(
                options=mice["supervised"],
                value="TX108",
                description="Supervised",
            ),

            "unsupervised_mouse": widgets.Dropdown(
                options=mice["unsupervised"],
                value="DR10",
                description="Unsupervised",
            ),

            "grating_mouse": widgets.Dropdown(
                options=mice["grating"],
                value="LZ13",
                description="Grating",
            ),

            "threshold": widgets.FloatSlider(
                value=0.3,
                min=0.05,
                max=1.0,
                step=0.05,
                description="|d′| threshold",
            ),
        },
        show="supervised_figure",
        auto_run=False,
    )
)

three_mouse_selectivity_panel

In [ ]:
mice_recordings

In [ ]:
mice_recordings['supervised']

In [ ]:
sup_train1_before_learning = {
    mouse: [
        recording for recording in recordings
        if 'sup_train1_before_learning' in recording.experiments
    ]
    for (mouse, recordings) in mice_recordings['supervised'].items()
}
sup_train1_before_learning

In [ ]:
sup_train1_before_learning = {
    k: v[0] for (k, v) in sup_train1_before_learning.items() if len(v) > 0
}
sup_train1_before_learning

In [ ]:
sup_train1_after_learning = {
    mouse: [
        recording for recording in recordings
        if 'sup_train1_after_learning' in recording.experiments
    ]
    for (mouse, recordings) in mice_recordings['supervised'].items()
}
sup_train1_after_learning

In [ ]:
sup_train1_after_learning = {
    k: v[0] for (k, v) in sup_train1_after_learning.items() if len(v) > 0
}
sup_train1_after_learning

In [ ]:
TX108_recording_before = sup_train1_before_learning['TX108']
inspect(TX108_recording_before)

In [ ]:
TX108_recording_after = sup_train1_after_learning['TX108']
inspect(TX108_recording_after)

In [ ]:
TX108_before = vars(TX108_recording_before)
TX108_before

In [ ]:
TX108_after = vars(TX108_recording_after)
TX108_after

In [ ]:
vars(TX108_before['files'][0])

In [ ]:
behavior_before_training = TX108_recording_before.load('behavior', experiment = 'sup_train1_before_learning')
behavior_before_training

In [ ]:
behavior_after_training = TX108_recording_after.load('behavior', experiment = 'sup_train1_after_learning')
behavior_after_training.keys()